In [ ]:
# cifar10 is basically images and each is one of ten classes labeled from 0 to 9

In [ ]:
from datasets import load_dataset
from flwr_datasets import FederatedDataset
from flwr_datasets.partitioner import IidPartitioner
from flwr_datasets.partitioner import PathologicalPartitioner
from torch.utils.data import DataLoader
from torchvision.transforms import Compose, Normalize, ToTensor

In [ ]:
pytorch_transforms = Compose([ToTensor(), Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))])

In [ ]:
fds = None

In [ ]:
def apply_transforms(batch):
    """Apply transforms to the partition from FederatedDataset."""
    batch["img"] = [pytorch_transforms(img) for img in batch["img"]]
    return batch

In [ ]:
def load_data(partition_id: int, num_partitions: int, batch_size: int):
    """Load partition CIFAR10 data."""
    # Only initialize `FederatedDataset` once
    global fds
    if fds is None:
        partitioner = IidPartitioner(num_partitions=num_partitions)
        fds = FederatedDataset(
            dataset="uoft-cs/cifar10",
            partitioners={"train": partitioner},
        )
    partition = fds.load_partition(partition_id)
    # Divide data on each node: 80% train, 20% test
    partition_train_test = partition.train_test_split(test_size=0.2, seed=42)
    # Construct dataloaders
    partition_train_test = partition_train_test.with_transform(apply_transforms)
    trainloader = DataLoader(
        partition_train_test["train"], batch_size=batch_size, shuffle=True
    )
    testloader = DataLoader(partition_train_test["test"], batch_size=batch_size)
    return trainloader, testloader

In [ ]:
def load_nonIID_data(partition_id: int, num_partitions: int, batch_size: int):
    """Load partition CIFAR10 data."""
    # Only initialize `FederatedDataset` once
    global fds
    if fds is None:
        partitioner = PathologicalPartitioner(num_partitions=num_partitions, partition_by="label", num_classes_per_partition=2, class_assignment_mode="first-deterministic")
        fds = FederatedDataset(
            dataset="uoft-cs/cifar10",
            partitioners={"train": partitioner},
        )
    partition = fds.load_partition(partition_id)
    # Divide data on each node: 80% train, 20% test
    partition_train_test = partition.train_test_split(test_size=0.2, seed=42)
    # Construct dataloaders
    partition_train_test = partition_train_test.with_transform(apply_transforms)
    trainloader = DataLoader(
        partition_train_test["train"], batch_size=batch_size, shuffle=True
    )
    testloader = DataLoader(partition_train_test["test"], batch_size=batch_size)
    return trainloader, testloader

In [ ]:
train_loader, test_loader = load_data(0, 10, 10)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/5.16k [00:00<?, ?B/s]

plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/120M [00:00<?, ?B/s]

plain_text/test-00000-of-00001.parquet:   0%|          | 0.00/23.9M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/50000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/10000 [00:00<?, ? examples/s]

In [ ]:
print(train_loader.dataset)

Dataset({
    features: ['img', 'label'],
    num_rows: 4000
})


In [ ]:
print(train_loader.dataset[2])

{'img': tensor([[[-0.2314, -0.2471, -0.2471,  ..., -0.5216, -0.5216, -0.5059],
         [-0.0980, -0.1137, -0.0980,  ..., -0.4431, -0.4431, -0.4353],
         [-0.0588, -0.0667, -0.0510,  ..., -0.2235, -0.2078, -0.2157],
         ...,
         [-0.4118, -0.2157, -0.1059,  ...,  0.0980,  0.1529,  0.2235],
         [-0.1922, -0.1922, -0.2627,  ...,  0.2157,  0.1294,  0.0902],
         [-0.0667, -0.0745, -0.1451,  ...,  0.1529,  0.0431, -0.0118]],

        [[-0.2549, -0.2784, -0.2706,  ..., -0.4667, -0.4667, -0.4588],
         [-0.1608, -0.1686, -0.1529,  ..., -0.4196, -0.4196, -0.4196],
         [-0.1608, -0.1686, -0.1529,  ..., -0.2471, -0.2314, -0.2549],
         ...,
         [-0.4980, -0.2941, -0.1843,  ...,  0.0431,  0.0980,  0.1608],
         [-0.2863, -0.2784, -0.3490,  ...,  0.1294,  0.0353, -0.0039],
         [-0.1608, -0.1608, -0.2314,  ...,  0.0431, -0.0667, -0.1216]],

        [[-0.4431, -0.4588, -0.4588,  ..., -0.5608, -0.5608, -0.5529],
         [-0.4824, -0.4824, -0.4745, 

In [ ]:
train_loader, test_loader = load_nonIID_data(0,10,10)

In [ ]:
print(train_loader.dataset)

Dataset({
    features: ['img', 'label'],
    num_rows: 2667
})


In [ ]:
print(train_loader.dataset[12])

{'img': tensor([[[0.4510, 0.5529, 0.5451,  ..., 0.6157, 0.6157, 0.5922],
         [0.5294, 0.5608, 0.5608,  ..., 0.6157, 0.6235, 0.6000],
         [0.5529, 0.5686, 0.5608,  ..., 0.6000, 0.6235, 0.6000],
         ...,
         [0.7020, 0.7020, 0.6784,  ..., 0.7882, 0.8039, 0.7647],
         [0.7333, 0.7333, 0.5843,  ..., 0.7961, 0.8039, 0.7882],
         [0.7569, 0.7647, 0.7098,  ..., 0.7804, 0.7569, 0.7333]],

        [[0.6000, 0.6863, 0.6784,  ..., 0.7176, 0.7333, 0.7098],
         [0.6627, 0.6863, 0.6863,  ..., 0.7333, 0.7412, 0.7176],
         [0.6784, 0.6784, 0.6863,  ..., 0.7333, 0.7412, 0.7176],
         ...,
         [0.5529, 0.5686, 0.5608,  ..., 0.6941, 0.7098, 0.6863],
         [0.5922, 0.6078, 0.4588,  ..., 0.7176, 0.7255, 0.7098],
         [0.6392, 0.6471, 0.5765,  ..., 0.7020, 0.6784, 0.6392]],

        [[0.6471, 0.7412, 0.7333,  ..., 0.7804, 0.7882, 0.7647],
         [0.7176, 0.7412, 0.7412,  ..., 0.7804, 0.7961, 0.7725],
         [0.7255, 0.7333, 0.7333,  ..., 0.7725, 0.